# Predictive Analytics for Early Detection of Mental Health Risks Using Lifestyle and Behavioral Data

**Dataset:** `Meantal_health.csv` (3000 records) — Kaggle: *Mental Health and Lifestyle Habits (2019-2024)*.
Columns: Country, Age, Gender, Exercise Level, Diet Type, Sleep Hours, Stress Level, Mental Health Condition,
Work Hours per Week, Screen Time per Day, Social Interaction Score, Happiness Score.

We convert `Mental Health Condition` (None / Anxiety / Depression / PTSD / Bipolar) into a binary target
`at_risk` (0 = None, 1 = any condition present) for early risk-screening.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (7, 4.5)
RANDOM_STATE = 42


## 1. Load Dataset


In [ ]:
df = pd.read_csv("Mental_health.csv")
df["at_risk"] = (df["Mental Health Condition"] != "None").astype(int)
print(df.shape)
df.head()


## 2. Data Exploration


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.describe()


In [ ]:
df["Mental Health Condition"].value_counts()


In [ ]:
df["at_risk"].value_counts(normalize=True)


## 3. Exploratory Data Analysis


In [ ]:
sns.countplot(data=df, x="at_risk", hue="at_risk", palette=["#4C72B0", "#DD8452"], legend=False)
plt.title("Target Class Distribution")
plt.xticks([0, 1], ["Not at risk", "At risk"])
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features = ["Sleep Hours", "Work Hours per Week", "Screen Time per Day (Hours)", "Social Interaction Score", "Happiness Score", "Age"]
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(data=df, x="at_risk", y=col, hue="at_risk", ax=ax, palette=["#4C72B0", "#DD8452"], legend=False)
    ax.set_xticklabels(["No", "Yes"])
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=df, x="Stress Level", hue="at_risk", ax=axes[0], palette=["#4C72B0", "#DD8452"], order=["Low", "Moderate", "High"])
axes[0].set_title("Risk by Stress Level")
sns.countplot(data=df, x="Exercise Level", hue="at_risk", ax=axes[1], palette=["#4C72B0", "#DD8452"], order=["Low", "Moderate", "High"])
axes[1].set_title("Risk by Exercise Level")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


## 4. Data Preprocessing

Encode the categorical columns (`Country`, `Gender`, `Exercise Level`, `Diet Type`, `Stress Level`),
drop the original `Mental Health Condition` column (it directly determines the target), split into
train/test sets, and scale the features.


In [ ]:
data = df.drop(columns=["Mental Health Condition"])
data = pd.get_dummies(
    data,
    columns=["Country", "Gender", "Exercise Level", "Diet Type", "Stress Level"],
    drop_first=True
)

X = data.drop(columns=["at_risk"])
y = data["at_risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

X_train.shape, X_test.shape


## 5. Model Building

Train and compare four classifiers.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

results = []
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
    })
    fitted[name] = model

results_df = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)
results_df


In [ ]:
results_df.set_index("Model").plot(kind="bar", figsize=(10, 5))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 6. Evaluation of Best Model


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
y_pred = best_model.predict(X_test)

print("Best model:", best_name)
print(classification_report(y_test, y_pred, target_names=["Not at risk", "At risk"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 7. Feature Importance


In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    importances.head(10).plot(kind="barh")
    plt.title("Top 10 Important Features")
    plt.gca().invert_yaxis()
    plt.show()


## 8. Conclusion

- The dataset shows only weak-to-moderate lifestyle correlations with the presence of a mental health
  condition (see correlation heatmap and boxplots above) — this is typical of real-world survey data, unlike
  the earlier synthetic version where relationships were engineered to be strong.
- Compare the model scores above: note which model generalizes best (highest F1 / balanced precision-recall)
  on this dataset, and report it in your write-up.
- This model is intended as an early-screening aid, not a clinical diagnosis — flagged individuals should be
  referred for professional evaluation.
